# Классификация тональности твитов о COVID-19

Проект определяет тональность англоязычных твитов с помощью двунаправленной LSTM и механизма внимания.

В проекте показан полный цикл работы: загрузка и очистка данных, исследование выборки, подготовка словаря, обучение, оценка на отложенной тестовой выборке и анализ ошибок.


## Схема эксперимента

- исходная обучающая выборка делится на `train` и `validation` со стратификацией;
- словарь строится только по `train`, чтобы избежать утечки данных;
- лучшая модель выбирается по `macro F1` на `validation`;
- тестовая выборка используется один раз — для итоговой оценки.


In [ ]:
import json
import random
import re
import time
import zipfile
from collections import Counter
from functools import partial
from pathlib import Path

import gdown
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import torch
from nltk.corpus import stopwords
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


In [ ]:
SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## Данные

Набор содержит твиты, размеченные по пяти классам тональности: от крайне негативной до крайне позитивной. Данные автоматически скачиваются в локальную папку `data/`, которая не попадает в Git.


In [ ]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
DATA_DIR.mkdir(exist_ok=True)
ARTIFACTS_DIR.mkdir(exist_ok=True)

TRAIN_CSV = DATA_DIR / "Corona_NLP_train.csv"
TEST_CSV = DATA_DIR / "Corona_NLP_test.csv"
ARCHIVE_PATH = DATA_DIR / "coronavirus_archive.zip"
GOOGLE_DRIVE_FILE_ID = "145ZCFU-b65Xbjsh16fPepF_N66X0kZj2"

if not TRAIN_CSV.exists() or not TEST_CSV.exists():
    if not ARCHIVE_PATH.exists():
        gdown.download(id=GOOGLE_DRIVE_FILE_ID, output=str(ARCHIVE_PATH), quiet=False)
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(DATA_DIR)

missing = [path.name for path in (TRAIN_CSV, TEST_CSV) if not path.exists()]
if missing:
    raise FileNotFoundError(f"Не найдены файлы: {missing}")

print(TRAIN_CSV)
print(TEST_CSV)


In [ ]:
USE_COLUMNS = ["OriginalTweet", "Sentiment"]
source_train = pd.read_csv(TRAIN_CSV, encoding="latin-1", usecols=USE_COLUMNS).dropna()
test_df = pd.read_csv(TEST_CSV, encoding="latin-1", usecols=USE_COLUMNS).dropna()

train_df, val_df = train_test_split(
    source_train,
    test_size=0.2,
    random_state=SEED,
    stratify=source_train["Sentiment"],
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"train: {len(train_df):,}")
print(f"validation: {len(val_df):,}")
print(f"test: {len(test_df):,}")
train_df.head()


## Очистка и токенизация

Ссылки и упоминания пользователей заменяются специальными токенами. Слова с отрицанием и усилители (`not`, `very`, `too` и другие) сохраняются: они важны для определения тональности.


In [ ]:
nltk.download("stopwords", quiet=True)

KEEP_WORDS = {
    "no", "nor", "not", "never", "very", "too", "more", "most",
    "less", "least", "against", "only", "but", "down", "up",
}
STOP_WORDS = set(stopwords.words("english")) - KEEP_WORDS
TOKEN_PATTERN = re.compile(r"_link|_user|[a-z]+(?:'[a-z]+)?")


def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " _link ", text)
    text = re.sub(r"@\w+", " _user ", text)
    text = text.replace("#", "")
    tokens = TOKEN_PATTERN.findall(text)
    return [token for token in tokens if token not in STOP_WORDS]


for frame in (train_df, val_df, test_df):
    frame["tokens"] = frame["OriginalTweet"].map(tokenize)
    frame.drop(frame[frame["tokens"].map(len) == 0].index, inplace=True)
    frame.reset_index(drop=True, inplace=True)

train_df[["OriginalTweet", "tokens"]].head(3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
train_df["Sentiment"].value_counts().sort_index().plot.bar(ax=axes[0], title="Классы в train")
train_df["tokens"].map(len).plot.hist(bins=40, ax=axes[1], title="Длина твитов после очистки")
axes[0].set_xlabel("Тональность")
axes[0].set_ylabel("Количество")
axes[1].set_xlabel("Количество токенов")
plt.tight_layout()
plt.show()

train_df["tokens"].map(len).describe(percentiles=[0.5, 0.9, 0.95, 0.99])


## Словарь и загрузчики данных


In [ ]:
MAX_VOCAB_SIZE = 15_000
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

counter = Counter(token for tokens in train_df["tokens"] for token in tokens)
vocabulary = [PAD_TOKEN, UNK_TOKEN] + [
    token for token, _ in counter.most_common(MAX_VOCAB_SIZE - 2)
]
token_to_id = {token: index for index, token in enumerate(vocabulary)}
PAD_ID = token_to_id[PAD_TOKEN]
UNK_ID = token_to_id[UNK_TOKEN]

classes = sorted(train_df["Sentiment"].unique())
label_to_id = {label: index for index, label in enumerate(classes)}
id_to_label = {index: label for label, index in label_to_id.items()}

print("Размер словаря:", len(vocabulary))
print("Классы:", classes)


In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, frame, token_to_id, label_to_id):
        self.tokens = frame["tokens"].tolist()
        self.labels = frame["Sentiment"].map(label_to_id).tolist()
        self.token_to_id = token_to_id

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        token_ids = [self.token_to_id.get(token, UNK_ID) for token in self.tokens[index]]
        return torch.tensor(token_ids, dtype=torch.long), self.labels[index]


def collate_batch(batch, pad_id):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(sequence) for sequence in sequences], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True, padding_value=pad_id)
    return padded, lengths, torch.tensor(labels, dtype=torch.long)


BATCH_SIZE = 64
loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": 0,
    "pin_memory": device.type == "cuda",
    "collate_fn": partial(collate_batch, pad_id=PAD_ID),
}

train_loader = DataLoader(
    SentimentDataset(train_df, token_to_id, label_to_id),
    shuffle=True,
    **loader_kwargs,
)
val_loader = DataLoader(SentimentDataset(val_df, token_to_id, label_to_id), **loader_kwargs)
test_loader = DataLoader(SentimentDataset(test_df, token_to_id, label_to_id), **loader_kwargs)


## Модель

BiLSTM читает текст в обоих направлениях. Механизм внимания присваивает вес каждому непустому токену и формирует единое представление твита. Маска не даёт модели учитывать добавленные при выравнивании `<pad>`-токены.


In [ ]:
class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, pad_id, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attention = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, tokens, lengths):
        embedded = self.embedding(tokens)
        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_output, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(packed_output, batch_first=True)

        scores = self.attention(output).squeeze(-1)
        positions = torch.arange(output.size(1), device=output.device).unsqueeze(0)
        padding_mask = positions >= lengths.to(output.device).unsqueeze(1)
        scores = scores.masked_fill(padding_mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(output * weights.unsqueeze(-1), dim=1)
        return self.classifier(self.dropout(context))


model = BiLSTMAttentionClassifier(
    vocab_size=len(vocabulary),
    embedding_dim=64,
    hidden_dim=128,
    num_classes=len(classes),
    pad_id=PAD_ID,
).to(device)

print(model)


## Обучение и выбор лучшей модели


In [ ]:
def run_epoch(model, dataloader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)
    running_loss = 0.0
    targets, predictions = [], []

    for tokens, lengths, labels in tqdm(dataloader, leave=False):
        tokens = tokens.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(tokens, lengths)
            loss = criterion(logits, labels)
            if is_training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        running_loss += loss.item() * labels.size(0)
        targets.extend(labels.cpu().tolist())
        predictions.extend(logits.argmax(dim=1).cpu().tolist())

    return {
        "loss": running_loss / len(dataloader.dataset),
        "accuracy": accuracy_score(targets, predictions),
        "macro_f1": f1_score(targets, predictions, average="macro"),
        "targets": targets,
        "predictions": predictions,
    }


In [ ]:
EPOCHS = 15
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
scheduler = StepLR(optimizer, step_size=4, gamma=0.6)
best_model_path = ARTIFACTS_DIR / "best_model.pt"
best_val_f1 = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    started_at = time.time()
    train_metrics = run_epoch(model, train_loader, criterion, optimizer)
    val_metrics = run_epoch(model, val_loader, criterion)
    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_macro_f1": val_metrics["macro_f1"],
    }
    history.append(row)

    if val_metrics["macro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["macro_f1"]
        torch.save(model.state_dict(), best_model_path)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {train_metrics['loss']:.4f}, F1 {train_metrics['macro_f1']:.4f} | "
        f"val loss {val_metrics['loss']:.4f}, F1 {val_metrics['macro_f1']:.4f} | "
        f"{time.time() - started_at:.1f}s"
    )


In [ ]:
history_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history_df.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0], title="Loss")
history_df.plot(x="epoch", y=["train_macro_f1", "val_macro_f1"], ax=axes[1], title="Macro F1")
plt.tight_layout()
plt.show()


## Итоговая оценка

Тестовая выборка используется только после выбора лучшей эпохи. Помимо accuracy рассчитывается macro F1, которая одинаково учитывает все пять классов.


In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))
test_metrics = run_epoch(model, test_loader, criterion)

print(f"Test accuracy: {test_metrics['accuracy']:.4f}")
print(f"Test macro F1: {test_metrics['macro_f1']:.4f}")
print(
    classification_report(
        test_metrics["targets"],
        test_metrics["predictions"],
        target_names=classes,
        digits=4,
    )
)

ConfusionMatrixDisplay.from_predictions(
    test_metrics["targets"],
    test_metrics["predictions"],
    display_labels=classes,
    xticks_rotation=35,
    cmap="Blues",
)
plt.tight_layout()
plt.show()

metrics_to_save = {
    "accuracy": test_metrics["accuracy"],
    "macro_f1": test_metrics["macro_f1"],
    "classes": classes,
}
(ARTIFACTS_DIR / "test_metrics.json").write_text(
    json.dumps(metrics_to_save, indent=2),
    encoding="utf-8",
)


In [ ]:
error_analysis = test_df[["OriginalTweet", "Sentiment"]].copy()
error_analysis["prediction"] = [id_to_label[index] for index in test_metrics["predictions"]]
error_analysis = error_analysis[error_analysis["Sentiment"] != error_analysis["prediction"]]

print("Ошибок:", len(error_analysis))
error_analysis.sample(min(15, len(error_analysis)), random_state=SEED)


## Выводы и следующие шаги

В исходном эксперименте модель получила `accuracy = 0.805` и `macro F1 = 0.8132`. Эти значения оставлены как исторический ориентир: в исходной версии тестовая выборка проверялась после каждой эпохи, поэтому результат нельзя считать полностью независимой итоговой оценкой.

В этой версии протокол исправлен: добавлена отдельная валидационная выборка, словарь строится только по обучающим данным, лучшая эпоха выбирается по validation macro F1, а тест используется один раз. После полного запуска сюда стоит внести новые итоговые значения из `artifacts/test_metrics.json`.

Возможные улучшения:

- сравнить модель с TF-IDF + Logistic Regression как простым базовым решением;
- попробовать предобученные эмбеддинги или Transformer;
- изучить ошибки между соседними классами тональности;
- добавить автоматические тесты для токенизации и подготовки батчей.
